# FLEX-Med — Standalone FL Simulation Notebook

**Single self-contained notebook** for running the entire Federated Learning pipeline on Google Colab.

Each code cell uses `%%writefile` to create the module files, then the API + ngrok cells run the simulation.

**Run all cells in order.** After ngrok starts, POST to the `/start_fl` endpoint to trigger FL.

---
## 1. Setup & Dependencies

In [ ]:
import sys
import os

from google.colab import drive
drive.mount('/content/drive')

# --- Configuration: Set your project path here ---
PROJECT_ROOT = "/content/drive/MyDrive/College/FLEX-Med/backend/federated_learning"

%cd {PROJECT_ROOT}

# Install dependencies
!pip install fastapi uvicorn pyngrok nest-asyncio 'flwr[simulation]==1.23.0' flwr_datasets datasets supabase scikit-learn -q

# Ensure package structure exists
os.makedirs("flex_med/utils", exist_ok=True)

# Add project root to path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

---
## 2. Configuration (`flex_med/utils/config.py`)

Standalone configuration — no dependency on `app.config`. All paths are resolved relative to the project root or overridden via environment variables.

In [ ]:
%%writefile flex_med/utils/__init__.py
# flex_med/utils package

In [ ]:
%%writefile flex_med/__init__.py
# flex_med package

In [ ]:
%%writefile flex_med/utils/config.py
"""Standalone configuration for FLEX-Med FL simulation (Colab-compatible)."""
import os
from pathlib import Path

# ============================================================================
# PATH RESOLUTION
# ============================================================================
_CURRENT_FILE = Path(__file__).resolve()
_FL_DIR = _CURRENT_FILE.parent.parent.parent  # flex_med/utils/config.py -> federated_learning/
_BACKEND_ROOT = _FL_DIR.parent  # federated_learning/ -> backend/

BASE_PATH = os.getenv("BASE_PATH", str(_BACKEND_ROOT))
PROJECT_DIR = str(_FL_DIR)

# ============================================================================
# LOGS
# ============================================================================
SIMULATION_LOG_PATH = os.getenv(
    "SIMULATION_LOG_PATH",
    str(_FL_DIR / "flex_med" / "utils" / "simulation_run.log")
)

# ============================================================================
# CLIENT CONFIGURATION
# ============================================================================
CLIENT_INFO_FILE_PATH = os.getenv(
    "FLEX_MED_CONFIG_FILE",
    str(_FL_DIR / "flex_med" / "utils" / "client_data.json")
)

# ============================================================================
# PYPROJECT.TOML
# ============================================================================
PYPROJECT_PATH = str(_FL_DIR / "pyproject.toml")

# ============================================================================
# DATASETS
# ============================================================================
DATASET_FILE_PATH = os.getenv(
    "DATASET_FILE_PATH",
    str(Path(BASE_PATH) / "datasets" / "cnmc")
)
PUBLIC_ANCHOR_DATASET_PATH = os.getenv(
    "PUBLIC_ANCHOR_DATASET_PATH",
    str(Path(DATASET_FILE_PATH) / "cnmc_public_anchor")
)
PUBLIC_TEST_DATASET_PATH = os.getenv(
    "PUBLIC_TEST_DATASET_PATH",
    str(Path(DATASET_FILE_PATH) / "cnmc_public_test")
)
LOCAL_TRAIN_DATASET_PATH = os.getenv(
    "LOCAL_TRAIN_DATASET_PATH",
    str(Path(DATASET_FILE_PATH) / "cnmc_local_train")
)

# ============================================================================
# MODEL CHECKPOINTS
# ============================================================================
MODEL_CHECKPOINT_FILE_PATH = os.getenv(
    "CHECKPOINTS_PATH",
    str(Path(BASE_PATH) / "checkpoints")
)

# ============================================================================
# METRICS & GRAPHS
# ============================================================================
GRAPHS_OUTPUT_DIR = os.getenv(
    "GRAPHS_OUTPUT_DIR",
    str(Path(BASE_PATH) / "graphical_visualisation")
)

# ============================================================================
# MODEL CONSTRAINTS
# ============================================================================
NUM_CLASSES = 2
IMG_SIZE = 256

# ============================================================================
# SIMULATION RUNNER
# ============================================================================
RUN_SIMULATION_SHELL_FILE_PATH = str(_FL_DIR / "flex_med" / "utils" / "run_simulation.sh")

# ============================================================================
# LOSS WEIGHTING
# ============================================================================
TRAIN_LOSS_WEIGHT = float(os.getenv("FLEX_MED_TRAIN_LOSS_WEIGHT", "0.72"))
DISTILL_LOSS_WEIGHT = float(os.getenv("FLEX_MED_DISTILL_LOSS_WEIGHT", "0.28"))

# ============================================================================
# DIRICHLET PARTITIONING
# ============================================================================
DIRICHLET_ALPHA = float(os.getenv("FLEX_MED_DIRICHLET_ALPHA", "1.5"))
DIRICHLET_SEED = int(os.getenv("FLEX_MED_DIRICHLET_SEED", "42"))
DIRICHLET_MIN_PARTITION_SIZE = int(os.getenv("FLEX_MED_DIRICHLET_MIN_PARTITION_SIZE", "400"))

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
MINORITY_BOOST = float(os.getenv("FLEX_MED_MINORITY_BOOST", "0.90"))

DISTILL_WEIGHT_BASE = float(os.getenv("FLEX_MED_DISTILL_WEIGHT", "0.80"))
DISTILL_DECAY_RATE = float(os.getenv("FLEX_MED_DISTILL_DECAY", "0.10"))

WEIGHT_DECAY = float(os.getenv("FLEX_MED_WEIGHT_DECAY", "0.02"))

---
## 3. Helper Utilities (`flex_med/utils/helpers.py`)

Client config loading, model creation, freeze/unfreeze strategy, database persistence.

In [ ]:
%%writefile flex_med/utils/helpers.py
import os
import json
from datetime import datetime
from typing import Dict, List, Optional
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights, EfficientNet_B1_Weights, EfficientNet_B2_Weights
from pathlib import Path
from flex_med.utils.config import (
    LOCAL_TRAIN_DATASET_PATH, DIRICHLET_ALPHA, DIRICHLET_SEED, CLIENT_INFO_FILE_PATH, NUM_CLASSES)

# <----------------------------- CONSTANTS & GLOBAL VARIABLES ----------------------------->
try:
    from supabase import create_client, Client
    SUPABASE_AVAILABLE = True
except ImportError:
    SUPABASE_AVAILABLE = False

def get_supabase_client() -> Optional['Client']:
    """Initialize Supabase client from environment variables."""
    if not SUPABASE_AVAILABLE:
        return None
    supabase_url = os.getenv("SUPABASE_URL", "https://sxwclekerhjlyxehaywu.supabase.co")
    supabase_key = os.getenv("SUPABASE_KEY", "sb_secret_bDFZ4Q4ATX0q2TmMWibD_A_1t4j6YT2")
    if not supabase_url or not supabase_key:
        return None
    try:
        return create_client(supabase_url, supabase_key)
    except Exception:
        return None

def get_simulation_id() -> Optional[int]:
    """Get simulation ID from environment or config file."""
    env_sim_id = os.getenv('FLEX_MED_SIMULATION_ID')
    if env_sim_id:
        try:
            return int(env_sim_id)
        except ValueError:
            pass

    try:
        env_config_path = os.getenv('FLEX_MED_CONFIG_FILE')
        if env_config_path and os.path.exists(env_config_path):
            with open(env_config_path, 'r') as f:
                config_data = json.load(f)
                if isinstance(config_data, dict) and "simulation_id" in config_data:
                    return config_data["simulation_id"]
    except Exception:
        pass
    return None

SUPABASE_CLIENT = get_supabase_client()
SIMULATION_ID = get_simulation_id()

# <----------------------------- PATH UTILITIES ----------------------------->
def sanitize_client_paths(clients: List[Dict]) -> List[Dict]:
    """Sanitize model paths to work in current environment."""
    backend_root = Path(__file__).parent.parent.parent
    for client in clients:
        if 'model_path' in client and client['model_path']:
            orig_path = client['model_path']
            if 'models' in orig_path:
                relative_part = orig_path.split('models')[-1].lstrip('/')
                client['model_path'] = str(backend_root / "models" / relative_part)
    return clients

# <----------------------------- CLIENT CONFIGURATION ----------------------------->
# Load client configuration from database.
def load_client_config(config_path: str = CLIENT_INFO_FILE_PATH) -> List[Dict]:
    if SUPABASE_CLIENT is not None and SIMULATION_ID is not None:
        try:
            # Loads the clients only relevant to this FL simulation
            response = SUPABASE_CLIENT.from_('client_simulation_metrics') \
                .select('client_id, clients(*)') \
                .eq('simulation_id', SIMULATION_ID) \
                .order('client_id') \
                .execute()
            if response.data:
                clients = [r.get('clients') for r in response.data if r.get('clients')]
                if clients:
                    return sanitize_client_paths(clients)
        except Exception:
            pass

    env_config_path = os.getenv('FLEX_MED_CONFIG_FILE')
    if env_config_path:
        config_path = env_config_path

    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Configuration file not found at {config_path}")

    with open(config_path, 'r') as f:
        config_data = json.load(f)

    clients = config_data.get("clients", config_data) if isinstance(config_data, dict) else config_data
    return sanitize_client_paths(clients)

# Retrieve individual client config by partition ID.
def get_client_by_partition_id(partition_id: int, config_path: str = CLIENT_INFO_FILE_PATH) -> Dict:
    from flwr.common import log
    from logging import WARNING

    clients = load_client_config(config_path)
    if partition_id >= len(clients):
        # Fallback: Cycle through available clients using modulo
        effective_id = partition_id % len(clients)
        log(WARNING, f"[WARNING] Partition ID {partition_id} exceeds clients ({len(clients)}). "
              f"Using client {effective_id} (Modulo fallback).")
        return clients[effective_id]
    return clients[partition_id]

def get_display_id(partition_id: int, client_config: Dict) -> str:
    """Get display ID for logging."""
    return client_config.get('client_name') or f"Client {client_config.get('id', partition_id + 1)}"

# <----------------------------- DATA HETEROGENEITY UTILITIES ----------------------------->
def get_partition_stats(partition_id: int, num_partitions: int, partitioner_cache: dict) -> Dict:
    """Get data heterogeneity statistics for a client's partition."""
    from flwr.common import log
    from logging import INFO, WARNING
    
    cache_key = (LOCAL_TRAIN_DATASET_PATH, num_partitions, DIRICHLET_ALPHA, DIRICHLET_SEED)

    # Try to find the partitioner in cache
    partitioner = None
    if cache_key in partitioner_cache:
        partitioner, _ = partitioner_cache[cache_key]
    else:
        # Try to find any matching cache entry (path might differ slightly)
        log(INFO, f"[DATA_HET] Exact cache key not found. Looking for alternatives...")
        log(INFO, f"[DATA_HET] Expected key: {cache_key}")
        log(INFO, f"[DATA_HET] Available keys: {list(partitioner_cache.keys())}")
        for key in partitioner_cache:
            if key[1] == num_partitions:  # Match by num_partitions
                partitioner, _ = partitioner_cache[key]
                log(INFO, f"[DATA_HET] Using fallback cache key: {key}")
                break
    
    if partitioner is None:
        log(WARNING, f"[DATA_HET] Partitioner not found in cache for {num_partitions} partitions")
        return {"error": "Partitioner not initialized"}

    try:
        partition_dataset = partitioner.load_partition(partition_id)
        partition_labels = partition_dataset["label"]

        total_samples = len(partition_labels)
        leukemia_count = sum(l == 0 for l in partition_labels)
        healthy_count = sum(l == 1 for l in partition_labels)

        leukemia_pct = leukemia_count / total_samples * 100 if total_samples > 0 else 0
        healthy_pct = healthy_count / total_samples * 100 if total_samples > 0 else 0

        max_class, min_class = max(leukemia_count, healthy_count), min(leukemia_count, healthy_count)
        imbalance_ratio = max_class / min_class if min_class > 0 else float('inf')

        log(INFO, f"[DATA_HET] Client {partition_id}: {total_samples} samples (L:{leukemia_count}, H:{healthy_count})")
        
        return {
            "total_samples": total_samples,
            "train_samples": int(total_samples * 0.85),
            "val_samples": total_samples - int(total_samples * 0.85),
            "class_distribution": {
                "leukemia": leukemia_count, "healthy": healthy_count,
                "leukemia_pct": round(leukemia_pct, 1), "healthy_pct": round(healthy_pct, 1)
            },
            "imbalance_ratio": round(imbalance_ratio, 2),
            "partition_id": partition_id
        }
    except Exception as e:
        log(WARNING, f"[DATA_HET] Error loading partition {partition_id}: {e}")
        return {"error": str(e)}

def save_all_clients_data_heterogeneity(client_configs: List[Dict], partitioner_cache: dict):
    """
    Save data heterogeneity statistics for all clients to the database.
    
    This function should be called at FL start to ensure data partitioning
    information is captured for visualization in the frontend.
    """
    from flwr.common import log
    from logging import INFO, WARNING
    
    if SUPABASE_CLIENT is None or SIMULATION_ID is None:
        log(WARNING, "[DATA_HET] Cannot save data heterogeneity: Supabase not initialized")
        return
    
    log(INFO, f"[DATA_HET] Saving data heterogeneity for {len(client_configs)} clients...")
    
    saved_count = 0
    for client_idx, config in enumerate(client_configs):
        db_client_id = config.get('id')
        if db_client_id is None:
            log(WARNING, f"[DATA_HET] Client {client_idx} has no database ID")
            continue
        
        try:
            # Get partition stats for this client
            stats = get_partition_stats(client_idx, len(client_configs), partitioner_cache)
            
            if "error" in stats:
                log(WARNING, f"[DATA_HET] Failed to get stats for client {client_idx}: {stats['error']}")
                continue
            
            # Fetch existing metrics
            response = SUPABASE_CLIENT.from_('client_simulation_metrics').select('metrics') \
                .eq('simulation_id', SIMULATION_ID).eq('client_id', db_client_id).execute()
            
            if not response.data:
                log(WARNING, f"[DATA_HET] No metrics record found for client {db_client_id}")
                continue
            
            existing_metrics = response.data[0].get('metrics', {})
            
            # Only update if data_heterogeneity not already present
            if 'data_heterogeneity' not in existing_metrics:
                existing_metrics['data_heterogeneity'] = stats
                
                SUPABASE_CLIENT.from_('client_simulation_metrics').update({'metrics': existing_metrics}) \
                    .eq('simulation_id', SIMULATION_ID).eq('client_id', db_client_id).execute()
                
                saved_count += 1
                log(INFO, f"[DATA_HET] Saved for client {client_idx}: {stats['total_samples']} samples "
                          f"(L:{stats['class_distribution']['leukemia']}, H:{stats['class_distribution']['healthy']})")
            else:
                log(INFO, f"[DATA_HET] Client {client_idx} already has data_heterogeneity")
                
        except Exception as e:
            log(WARNING, f"[DATA_HET] Error saving data heterogeneity for client {client_idx}: {e}")
    
    log(INFO, f"[DATA_HET] Saved data heterogeneity for {saved_count}/{len(client_configs)} clients")

def print_client_data_distribution_summary(client_configs: List[Dict], partitioner_cache: dict):
    """Print data distribution summary across all clients."""
    from flwr.common import log
    from logging import INFO

    log(INFO, "")
    log(INFO, "=" * 90)
    log(INFO, "CLIENT DATA DISTRIBUTION SUMMARY")
    log(INFO, "=" * 90)

    cache_key = (LOCAL_TRAIN_DATASET_PATH, len(client_configs), DIRICHLET_ALPHA, DIRICHLET_SEED)

    if cache_key in partitioner_cache:
        partitioner, _ = partitioner_cache[cache_key]
        total_all, total_healthy = 0, 0

        for i, config in enumerate(client_configs):
            client_name = config.get('client_name', f'Client {i}')
            partition_labels = partitioner.load_partition(i)["label"]
            all_count = sum(l == 0 for l in partition_labels)
            healthy_count = sum(l == 1 for l in partition_labels)
            total_samples = len(partition_labels)
            all_pct = all_count / total_samples * 100 if total_samples > 0 else 0
            healthy_pct = healthy_count / total_samples * 100 if total_samples > 0 else 0
            ratio = all_count / healthy_count if healthy_count > 0 else float('inf')

            total_all += all_count
            total_healthy += healthy_count

            log(INFO, f"Client {i} ({client_name}): Total={total_samples:4d} | "
                      f"ALL={all_count:4d} ({all_pct:5.1f}%) | Healthy={healthy_count:4d} ({healthy_pct:5.1f}%) | "
                      f"Ratio={ratio:5.1f}:1")

        overall_total = total_all + total_healthy
        overall_ratio = total_all / total_healthy if total_healthy > 0 else float('inf')
        log(INFO, "-" * 90)
        log(INFO, f"OVERALL: Total={overall_total:4d} | Ratio={overall_ratio:5.1f}:1 | "
                  f"Dirichlet alpha={DIRICHLET_ALPHA}, seed={DIRICHLET_SEED}")
        log(INFO, "=" * 90)
    else:
        log(INFO, "Partitioner not initialized yet")

# <----------------------------- MODEL UTILITIES ----------------------------->
def get_model_by_type(model_type: str, use_pretrained: bool = True, dropout_rate: float = None):
    """Create a model instance by type with dropout-enhanced classifier."""
    model_type = model_type.lower()
    if dropout_rate is None:
        dropout_rate = get_initial_dropout_rate(model_type)

    model_map = {
        'efficientnet_b0': (models.efficientnet_b0, EfficientNet_B0_Weights.DEFAULT),
        'efficientnet_b1': (models.efficientnet_b1, EfficientNet_B1_Weights.DEFAULT),
        'efficientnet_b2': (models.efficientnet_b2, EfficientNet_B2_Weights.DEFAULT),
    }

    if model_type not in model_map:
        raise ValueError(f"Unsupported model type: {model_type}. Supported: {list(model_map.keys())}")

    model_fn, default_weights = model_map[model_type]
    weights = default_weights if use_pretrained else None

    model = model_fn(weights=weights)
    return add_dropout_to_classifier(model, model_type, dropout_rate)

def get_initial_dropout_rate(model_type: str) -> float:
    """Get default dropout rate for model architecture."""
    rates = {'efficientnet_b0': 0.40, 'efficientnet_b1': 0.40, 'efficientnet_b2': 0.45}
    return rates.get(model_type.lower(), 0.3)

def add_dropout_to_classifier(model, model_type: str, dropout_rate: float = 0.3):
    """Add/update dropout layer before the final classifier head (idempotent)."""
    model_type = model_type.lower()

    match model_type:
        case 'efficientnet_b0' | 'efficientnet_b1' | 'efficientnet_b2':
            if isinstance(model.classifier[1], nn.Sequential) and isinstance(model.classifier[1][0], nn.Dropout):
                model.classifier[1][0].p = dropout_rate
            else:
                in_features = model.classifier[1].in_features
                model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(in_features, NUM_CLASSES))

    return model

def load_model_weights(model, model_path, device):
    """Load model weights from file (handles legacy and new formats)."""
    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    except Exception:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)
    return model

def save_model(model: torch.nn.Module, model_path: str, model_type: str):
    """Save model weights with minimal metadata."""
    torch.save({
        'model_type': model_type,
        'num_classes': NUM_CLASSES,
        'state_dict': model.state_dict(),
    }, model_path)

def load_model_for_client(partition_id: int, config_path: str = CLIENT_INFO_FILE_PATH):
    """Load client config and create model architecture."""
    client_config = get_client_by_partition_id(partition_id, config_path)
    model = get_model_by_type(client_config['model_type'])
    return model, client_config['model_path'], client_config

# <----------------------------- MODEL EVALUATION & TRAINING UTILITIES ----------------------------->
def compute_per_class_accuracy(model, valloader, device):
    """
    Compute per-class accuracies for balanced accuracy metric.
    """
    model.eval()
    class_correct = {0: 0, 1: 0}
    class_total = {0: 0, 1: 0}

    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            for i in range(labels.size(0)):
                label = labels[i].item()
                class_total[label] += 1
                if predicted[i] == label:
                    class_correct[label] += 1

    # Handle edge case: if one class has zero samples in validation set
    if class_total[0] == 0 or class_total[1] == 0:
        # Fall back to total accuracy
        total_correct = class_correct[0] + class_correct[1]
        total_samples = class_total[0] + class_total[1]
        return total_correct / total_samples if total_samples > 0 else 0.5

    # Compute per-class accuracies
    leukemia_acc = class_correct[0] / class_total[0]
    healthy_acc = class_correct[1] / class_total[1]

    # Return only balanced accuracy (privacy-preserving)
    balanced_accuracy = (leukemia_acc + healthy_acc) / 2.0

    return balanced_accuracy

# Freeze all backbone layers, keeping only the classifier trainable.
# Use in early FL rounds to prevent biased gradients from corrupting pretrained features.
def freeze_backbone(model, model_type: str):
    model_type = model_type.lower()

    # First, freeze all parameters
    for param in model.parameters():
        param.requires_grad = False

    # Then unfreeze only the classifier head
    if model_type.startswith('efficientnet'):
        for param in model.classifier.parameters():
            param.requires_grad = True

    return model

def set_bn_eval(m):
    """Freeze running statistics and parameters for ALL types of BatchNorm layers."""
    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
        m.eval()
        if m.weight is not None:
            m.weight.requires_grad = False
        if m.bias is not None:
            m.bias.requires_grad = False

# Progressive unfreezing: unfreeze fractions of the final backbone block (25% -> 50% -> 100%).
def unfreeze_fraction_of_last_block(model, model_type: str, fraction: float):
    """Unfreeze a specific fraction of the final block of the architecture."""
    model_type = model_type.lower()
    last_block = None

    if model_type.startswith('efficientnet'):
        for module in model.features[-1].modules():
            if not isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                for param in module.parameters(recurse=False):
                    param.requires_grad = True
        last_block = model.features[-2]  # Stage 7 (last MBConv stage)

    if last_block is not None:
        children = list(last_block.children())
        num_children = len(children)

        num_unfreeze = int(num_children * fraction)
        if fraction > 0.0 and num_unfreeze == 0:
            num_unfreeze = 1

        unfreeze_start_idx = num_children - num_unfreeze

        for i, child in enumerate(children):
            if i >= unfreeze_start_idx:
                for module in child.modules():
                    # Skip BatchNorm layers to keep statistics stable
                    if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                        continue
                    # Enable gradients for other parameters (Convs, Linear, etc)
                    for param in module.parameters(recurse=False):
                        param.requires_grad = True

    return model

PHASE2_START_ROUND = 5  # Phase 1: R1-4

def apply_freeze_strategy(model, model_type: str, server_round: int, total_rounds: int = 10):
    """2-phase progressive unfreezing strategy (Testing).
    Phase 1 (R1-4): Classifier head only — backbone frozen.
    Phase 2 (R5+): ~50% of final backbone block unfrozen (1 MBConv module).
    """
    from flwr.common import log
    from logging import INFO

    # Always start by freezing everything and unfreezing the classifier head
    model = freeze_backbone(model, model_type)

    if server_round < PHASE2_START_ROUND:
        # Phase 1: Head Only Warm-up
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log(INFO, f"[Freeze] Round {server_round}/{total_rounds}: Stage 1 - Classifier only ({trainable_params:,} params)")
    else:
        # Phase 2: 50% Backbone Refinement
        model = unfreeze_fraction_of_last_block(model, model_type, fraction=0.50)
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log(INFO, f"[Freeze] Round {server_round}/{total_rounds}: Stage 2 - 50% backbone block ({trainable_params:,} params)")

    return model

# <----------------------------- DATABASE PERSISTENCE & MONITORING UTILITIES ----------------------------->
def extract_training_metrics_for_persistence(client_metrics_list: List[Dict], aggregation_metadata: Dict, client_configs: List[Dict]) -> Dict:
    """Extract training metrics for persistence."""
    training_metrics = {}
    weight_breakdown = aggregation_metadata.get("weight_breakdown", [])
    for i, metrics in enumerate(client_metrics_list):
        consensus_weight = weight_breakdown[i].get("normalized_weight") if i < len(weight_breakdown) else None
        training_metrics[str(i)] = {
            "distill_loss": metrics.get("distill_loss"), "train_loss": metrics.get("train_loss"),
            "train_accuracy": metrics.get("train_accuracy"), "val_loss": metrics.get("val_loss"),
            "num-examples": metrics.get("num-examples"), "training_time": metrics.get("training_time"),
            "consensus_weight": consensus_weight
        }
    return training_metrics

def save_global_post_fl_metrics(metrics: Dict, client_configs: List[Dict], best_round: int = 0):
    """Save Post-FL metrics to database and compute aggregates."""
    if SUPABASE_CLIENT is None or SIMULATION_ID is None:
        return
    try:
        BASELINE = {'accuracy': 0.5, 'precision': 0.5, 'recall': 0.0, 'f1_score': 0.0, 'loss': 0.693,
                    'specificity': 0.5, 'roc_auc': 0.5, 'healthy_accuracy': 0.5, 'leukemia_accuracy': 0.5, 'class_gap': 1.0}

        for client_id_str, client_metrics in metrics.items():
            client_idx = int(client_id_str)
            if client_idx >= len(client_configs):
                continue
            db_client_id = client_configs[client_idx].get('id')
            if db_client_id is None:
                continue

            response = SUPABASE_CLIENT.from_('client_simulation_metrics').select('metrics') \
                .eq('simulation_id', SIMULATION_ID).eq('client_id', db_client_id).execute()
            if not response.data:
                continue

            existing_metrics = response.data[0].get('metrics', {})
            if 'global' not in existing_metrics:
                existing_metrics['global'] = {}
            existing_metrics['global']['post_fl'] = client_metrics
            existing_metrics['global']['improvement'] = {m: client_metrics[m] - b for m, b in BASELINE.items() if m in client_metrics}

            SUPABASE_CLIENT.from_('client_simulation_metrics').update({
                'metrics': existing_metrics, 'status': 'completed', 'completed_at': datetime.now().isoformat()
            }).eq('simulation_id', SIMULATION_ID).eq('client_id', db_client_id).execute()

        response = SUPABASE_CLIENT.from_('client_simulation_metrics').select('metrics').eq('simulation_id', SIMULATION_ID).execute()
        if response.data:
            aggregate_metrics = compute_aggregate_metrics_local([row['metrics'] for row in response.data])

            # Persist the best model round into aggregate metrics
            if best_round > 0:
                aggregate_metrics['best_model_round'] = best_round

            sim_response = SUPABASE_CLIENT.from_('fl_simulations').select('started_at').eq('id', SIMULATION_ID).execute()
            duration = None
            if sim_response.data and sim_response.data[0].get('started_at'):
                from datetime import timezone
                started_at = datetime.fromisoformat(sim_response.data[0]['started_at'].replace('Z', '+00:00'))
                duration = int((datetime.now(timezone.utc) - started_at).total_seconds())

            update_data = {'aggregate_metrics': aggregate_metrics, 'status': 'completed', 'completed_at': datetime.now().isoformat()}
            if duration:
                update_data['duration'] = duration
            SUPABASE_CLIENT.from_('fl_simulations').update(update_data).eq('id', SIMULATION_ID).execute()
    except Exception:
        pass

def compute_aggregate_metrics_local(all_metrics: List[dict]) -> dict:
    """Compute aggregate metrics from all client metrics."""
    if not all_metrics:
        return {}

    def calc_avg(metric_name: str, stage: str) -> float:
        values = [m['global'][stage].get(metric_name) for m in all_metrics
                  if stage in m.get('global', {}) and m['global'][stage].get(metric_name) is not None]
        return sum(values) / len(values) if values else 0.0

    def calc_std(metric_name: str, stage: str) -> float:
        values = [m['global'][stage].get(metric_name) for m in all_metrics
                  if stage in m.get('global', {}) and m['global'][stage].get(metric_name) is not None]
        if len(values) < 2:
            return 0.0
        mean = sum(values) / len(values)
        return (sum((x - mean) ** 2 for x in values) / len(values)) ** 0.5

    rounds_aggregate = []
    num_rounds = max((len(m.get('rounds', [])) for m in all_metrics), default=0)

    for round_num in range(1, num_rounds + 1):
        round_metrics = {'round': round_num, 'avg_accuracy': 0.0, 'avg_loss': 0.0, 'avg_f1': 0.0,
                         'avg_precision': 0.0, 'avg_recall': 0.0, 'avg_train_loss': 0.0, 'avg_val_loss': 0.0,
                         'num_clients_trained': 0, 'timestamp': None}

        val_data, train_data = [], []
        for m in all_metrics:
            for r in m.get('rounds', []):
                if r.get('round') == round_num:
                    if 'validation' in r:
                        val_data.append(r['validation'])
                        if not round_metrics['timestamp']:
                            round_metrics['timestamp'] = r['validation'].get('evaluated_at')
                    if 'training' in r:
                        train_data.append(r['training'])

        if val_data:
            round_metrics.update({
                'avg_accuracy': sum(v.get('accuracy', 0) for v in val_data) / len(val_data),
                'avg_loss': sum(v.get('loss', 0) for v in val_data) / len(val_data),
                'avg_f1': sum(v.get('f1_score', 0) for v in val_data) / len(val_data),
                'avg_precision': sum(v.get('precision', 0) for v in val_data) / len(val_data),
                'avg_recall': sum(v.get('recall', 0) for v in val_data) / len(val_data),
                'num_clients_trained': len(val_data)
            })

        train_losses = [t['train_loss'] for t in train_data if t.get('train_loss') is not None]
        val_losses = [t['val_loss'] for t in train_data if t.get('val_loss') is not None]
        if train_losses:
            round_metrics['avg_train_loss'] = sum(train_losses) / len(train_losses)
        if val_losses:
            round_metrics['avg_val_loss'] = sum(val_losses) / len(val_losses)

        if val_data or train_data:
            rounds_aggregate.append(round_metrics)

    best_round = max(rounds_aggregate, key=lambda r: r['avg_accuracy']) if rounds_aggregate else {}
    round_1 = next((r for r in rounds_aggregate if r['round'] == 1), {})
    final = rounds_aggregate[-1] if rounds_aggregate else {}

    improvement = {}
    if round_1 and final:
        for m in ['avg_accuracy', 'avg_loss', 'avg_f1', 'avg_precision', 'avg_recall']:
            if round_1.get(m) is not None and final.get(m) is not None:
                improvement[m] = round(final[m] - round_1[m], 6)

    return {
        "aggregate": {
            "post_fl": {"avg_accuracy": calc_avg('accuracy', 'post_fl'), "avg_loss": calc_avg('loss', 'post_fl'),
                        "avg_precision": calc_avg('precision', 'post_fl'), "avg_recall": calc_avg('recall', 'post_fl'),
                        "avg_f1": calc_avg('f1_score', 'post_fl'), "std_accuracy": calc_std('accuracy', 'post_fl'),
                        "num_clients": len(all_metrics)},
            "improvement": improvement
        },
        "rounds": rounds_aggregate,
        "best_round": {"round": best_round.get('round'), "avg_accuracy": best_round.get('avg_accuracy')} if best_round else {},
        "total_rounds_completed": len(rounds_aggregate), "total_clients": len(all_metrics)
    }

def save_round_training_metrics(round_num: int, training_metrics: Dict, client_configs: List[Dict], partitioner_cache: dict):
    """Save per-round training metrics to database."""
    if SUPABASE_CLIENT is None or SIMULATION_ID is None:
        return
    try:
        for client_id_str, client_metrics in training_metrics.items():
            client_idx = int(client_id_str)
            if client_idx >= len(client_configs):
                continue
            db_client_id = client_configs[client_idx].get('id')
            if db_client_id is None:
                continue

            response = SUPABASE_CLIENT.from_('client_simulation_metrics').select('metrics') \
                .eq('simulation_id', SIMULATION_ID).eq('client_id', db_client_id).execute()
            if not response.data:
                continue

            existing_metrics = response.data[0].get('metrics', {})
            if 'rounds' not in existing_metrics:
                existing_metrics['rounds'] = []

            round_entry = next((r for r in existing_metrics['rounds'] if r.get('round') == round_num), None)
            if round_entry is None:
                round_entry = {'round': round_num}
                existing_metrics['rounds'].append(round_entry)

            round_entry['training'] = {
                'train_loss': client_metrics.get('train_loss'), 'train_accuracy': client_metrics.get('train_accuracy'),
                'val_loss': client_metrics.get('val_loss'), 'distill_loss': client_metrics.get('distill_loss'),
                'training_time': client_metrics.get('training_time'), 'num_examples': client_metrics.get('num-examples'),
                'consensus_weight': client_metrics.get('consensus_weight')
            }

            if round_num == 1 and 'data_heterogeneity' not in existing_metrics:
                from flwr.common import log
                from logging import INFO, WARNING
                log(INFO, f"[DATA_HET] Attempting to save data_heterogeneity for client_idx={client_idx}")
                try:
                    stats = get_partition_stats(client_idx, len(client_configs), partitioner_cache)
                    if "error" not in stats:
                        existing_metrics['data_heterogeneity'] = stats
                        log(INFO, f"[DATA_HET] Successfully added data_heterogeneity for client {client_idx}")
                    else:
                        log(WARNING, f"[DATA_HET] get_partition_stats returned error: {stats}")
                except Exception as e:
                    log(WARNING, f"[DATA_HET] Exception while getting partition stats: {e}")

            SUPABASE_CLIENT.from_('client_simulation_metrics').update({'metrics': existing_metrics}) \
                .eq('simulation_id', SIMULATION_ID).eq('client_id', db_client_id).execute()
    except Exception as e:
        from flwr.common import log
        from logging import WARNING
        log(WARNING, f"[METRICS] Error saving round training metrics: {e}")

def save_round_validation_metrics(round_num: int, metrics: Dict, client_configs: List[Dict]):
    """Save per-round validation metrics to database."""
    if SUPABASE_CLIENT is None or SIMULATION_ID is None:
        return
    try:
        for client_id_str, client_metrics in metrics.items():
            client_idx = int(client_id_str)
            if client_idx >= len(client_configs):
                continue
            db_client_id = client_configs[client_idx].get('id')
            if db_client_id is None:
                continue

            response = SUPABASE_CLIENT.from_('client_simulation_metrics').select('metrics') \
                .eq('simulation_id', SIMULATION_ID).eq('client_id', db_client_id).execute()
            if not response.data:
                continue

            existing_metrics = response.data[0].get('metrics', {})
            if 'rounds' not in existing_metrics:
                existing_metrics['rounds'] = []

            round_entry = next((r for r in existing_metrics['rounds'] if r.get('round') == round_num), None)
            if round_entry is None:
                round_entry = {'round': round_num}
                existing_metrics['rounds'].append(round_entry)

            round_entry['validation'] = client_metrics

            SUPABASE_CLIENT.from_('client_simulation_metrics').update({'metrics': existing_metrics}) \
                .eq('simulation_id', SIMULATION_ID).eq('client_id', db_client_id).execute()
    except Exception:
        pass

def check_for_degradation_warnings(current_metrics: Dict, round_num: int, client_history: Dict):
    """Check validation metrics and log warnings if degradation detected."""
    from flwr.common import log
    from logging import WARNING

    if round_num < 3:
        return

    for client_id, history in client_history.items():
        if len(history) >= 3:
            last_3 = history[-3:]

            accs = []
            for r in last_3:
                val = r['metrics'].get('accuracy')
                accs.append(float(val) if val is not None and isinstance(val, (int, float)) else 0.0)

            if all(a is not None for a in accs):
                try:
                    if accs[-1] < accs[-2] < accs[-3]:
                        log(WARNING, f"Client {client_id}: Accuracy declining for 2 consecutive rounds")
                except TypeError:
                    pass

            losses = []
            for r in last_3:
                val = r['metrics'].get('loss')
                losses.append(float(val) if val is not None and isinstance(val, (int, float)) else 999.0)

            if all(l is not None for l in losses):
                try:
                    if losses[-1] > losses[-2] > losses[-3]:
                        log(WARNING, f"Client {client_id}: Loss increasing for 2 consecutive rounds (overfitting)")
                except TypeError:
                    pass


---
## 4. Core FL Logic (`flex_med/task.py`)

Data loaders, training, distillation, consensus computation, FLEXMedStrategy.

In [ ]:
%%writefile flex_med/task.py
import torch
import numpy as np
import os
import json
import time
import uuid
import shutil
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.transforms import Compose, ToTensor, Normalize
from typing import Tuple, Optional, Iterable, Dict, List
from flwr.common import Message, Metadata, RecordDict, ArrayRecord, ConfigRecord, log
from logging import INFO, WARNING
from flwr.serverapp.strategy import Strategy
from flwr.server import Grid
from flwr_datasets.partitioner import DirichletPartitioner
from datasets import Dataset
from flex_med.utils.config import (
    CLIENT_INFO_FILE_PATH,
    PUBLIC_ANCHOR_DATASET_PATH, PUBLIC_TEST_DATASET_PATH, LOCAL_TRAIN_DATASET_PATH,
    NUM_CLASSES, IMG_SIZE,
    DIRICHLET_ALPHA, DIRICHLET_SEED, DIRICHLET_MIN_PARTITION_SIZE,
    MINORITY_BOOST, WEIGHT_DECAY, DISTILL_WEIGHT_BASE, DISTILL_DECAY_RATE
)
from flex_med.utils.helpers import (
    sanitize_client_paths,
    get_partition_stats, print_client_data_distribution_summary,
    extract_training_metrics_for_persistence, save_global_post_fl_metrics,
    save_round_training_metrics, save_round_validation_metrics,
    check_for_degradation_warnings, SUPABASE_CLIENT, SIMULATION_ID,
    save_all_clients_data_heterogeneity,
    load_client_config, get_client_by_partition_id,
    get_model_by_type, load_model_weights, set_bn_eval
)

# <----------------------------- CONSTANTS & GLOBAL VARIABLES ----------------------------->
PARTITIONER_CACHE = {}

# <----------------------------- DATA TRANSFORMS ----------------------------->
COMMON_TRANSFORM = Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

PRIVATE_TRAIN_TRANSFORM = Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), # Standardize resolution
    transforms.RandomHorizontalFlip(p=0.5), # Handle arbitrary orientation
    transforms.RandomVerticalFlip(p=0.5),   # Cells aren't axis-aligned
    transforms.RandomRotation(15),          # Slight angular variations
    ToTensor(), # Convert to tensor
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet standards
])

class TransformOverrideSubset(torch.utils.data.Dataset):
    """Wraps a nested Subset to apply a different transform (e.g. no augmentation for val)."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
        # Resolve the root ImageFolder through nested Subsets
        ds = subset
        self._index_chain = []
        while isinstance(ds, torch.utils.data.Subset):
            self._index_chain.append(ds.indices)
            ds = ds.dataset
        self._root_dataset = ds  # The ImageFolder

    def _resolve_index(self, idx):
        resolved = idx
        for indices in self._index_chain:
            resolved = indices[resolved]
        return resolved

    def __getitem__(self, idx):
        root_idx = self._resolve_index(idx)
        path, label = self._root_dataset.samples[root_idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

    def __len__(self):
        return len(self.subset)

# <----------------------------- LOAD DATA ----------------------------->
# Create a Dirichlet partitioner for heterogeneous data distribution.
def create_dirichlet_partitioner(
    dataset_path: str, num_partitions: int, alpha: float = DIRICHLET_ALPHA,
    seed: int = DIRICHLET_SEED, min_partition_size: int = DIRICHLET_MIN_PARTITION_SIZE
) -> Tuple[DirichletPartitioner, datasets.ImageFolder]:
    full_dataset = datasets.ImageFolder(root=dataset_path, transform=None)
    image_paths = [path for path, _ in full_dataset.samples]
    labels = [label for _, label in full_dataset.samples]

    hf_dataset = Dataset.from_dict({"image_path": image_paths, "label": labels})
    partitioner = DirichletPartitioner(
        num_partitions=num_partitions, partition_by="label", alpha=alpha,
        min_partition_size=min_partition_size, self_balancing=False, shuffle=True, seed=seed
    )
    partitioner.dataset = hf_dataset
    return partitioner, full_dataset

# Load the public anchor dataset for consensus generation.
def load_public_dataset(batch_size=32):
    if not os.path.exists(PUBLIC_ANCHOR_DATASET_PATH):
        raise FileNotFoundError(f"Public data not found at {PUBLIC_ANCHOR_DATASET_PATH}")

    full_dataset = datasets.ImageFolder(root=PUBLIC_ANCHOR_DATASET_PATH, transform=COMMON_TRANSFORM)
    generator = torch.Generator().manual_seed(42)
    indices = torch.randperm(len(full_dataset), generator=generator).tolist()
    subset = torch.utils.data.Subset(full_dataset, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=2)

# Load and assign the data distribution to client models using Dirichlet partitioning.
def load_private_dataset(partition_id: int, num_partitions: int, batch_size=32,
                         config_path: str = CLIENT_INFO_FILE_PATH):
    if not os.path.exists(LOCAL_TRAIN_DATASET_PATH):
        raise FileNotFoundError(f"Shared training dataset not found at {LOCAL_TRAIN_DATASET_PATH}")

    cache_key = (LOCAL_TRAIN_DATASET_PATH, num_partitions, DIRICHLET_ALPHA, DIRICHLET_SEED)

    if cache_key not in PARTITIONER_CACHE:
        partitioner, full_dataset = create_dirichlet_partitioner(
            LOCAL_TRAIN_DATASET_PATH, num_partitions, DIRICHLET_ALPHA, DIRICHLET_SEED
        )
        PARTITIONER_CACHE[cache_key] = (partitioner, full_dataset)
    else:
        partitioner, full_dataset = PARTITIONER_CACHE[cache_key]

    partition_dataset = partitioner.load_partition(partition_id)
    partition_paths = partition_dataset["image_path"]

    path_to_idx = {path: idx for idx, (path, _) in enumerate(full_dataset.samples)}
    client_indices = [path_to_idx[path] for path in partition_paths]

    client_dataset = torch.utils.data.Subset(
        datasets.ImageFolder(root=LOCAL_TRAIN_DATASET_PATH, transform=PRIVATE_TRAIN_TRANSFORM),
        client_indices
    )

    # Stratified train/validation split (85/15)
    class_indices = {0: [], 1: []}
    for subset_idx, global_idx in enumerate(client_indices):
        class_indices[full_dataset.targets[global_idx]].append(subset_idx)

    train_indices, val_indices = [], []
    generator = torch.Generator().manual_seed(42)

    for indices in class_indices.values():
        if not indices:
            continue
        perm = torch.randperm(len(indices), generator=generator)
        shuffled = torch.tensor(indices)[perm].tolist()
        split = int(0.85 * len(shuffled))
        train_indices.extend(shuffled[:split])
        val_indices.extend(shuffled[split:])

    train_ds = torch.utils.data.Subset(client_dataset, train_indices)
    test_ds = TransformOverrideSubset(
        torch.utils.data.Subset(client_dataset, val_indices), COMMON_TRANSFORM
    )

    train_labels = [full_dataset.targets[client_indices[i]] for i in train_indices]
    class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

    class_weights = 1.0 / np.maximum(class_counts, 1) ** MINORITY_BOOST
    sample_weights = [class_weights[label] for label in train_labels]

    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    trainloader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2)
    testloader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    return trainloader, testloader, class_counts

# Load public test dataset for post_fl evaluation.
def load_public_test_dataset(batch_size=32):
    if not os.path.exists(PUBLIC_TEST_DATASET_PATH):
        raise FileNotFoundError(f"Public test data not found at {PUBLIC_TEST_DATASET_PATH}")
    dataset = datasets.ImageFolder(root=PUBLIC_TEST_DATASET_PATH, transform=COMMON_TRANSFORM)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Generate logits on the public anchor dataset for consensus computation.
def get_public_logits(model, public_loader, device):
    if public_loader is None:
        raise ValueError("public_loader is None.")
    model.to(device)
    model.eval()
    all_logits = []
    with torch.no_grad():
        for images, _ in public_loader:
            images = images.to(device)
            outputs = model(images)
            all_logits.append(outputs.cpu().numpy())
    return np.concatenate(all_logits)

# <----------------------------- LOCAL MODEL Training : Train client model with gradual unfreeze strategy. ----------------------------->
# Validation function during local training
def validate_training(model, valloader, criterion, device):
    model.eval()
    val_loss, val_batches = 0.0, 0
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            val_batches += 1
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = val_loss / val_batches if val_batches > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
    return avg_loss, accuracy

def train(model, trainloader, epochs, classifier_lr, backbone_lr, device,
          valloader=None, server_round: int = 1, total_rounds: int = 10):

    if trainloader is None:
        return 0.0, 0.0, 0.0, 0.0

    model.to(device)

    # Standard CrossEntropyLoss with label smoothing (WRS handles class balance)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    classifier_decay = []
    classifier_no_decay = []
    backbone_decay = []
    backbone_no_decay = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
            
        is_classifier = "classifier" in name or "fc" in name
        if is_classifier:
            if len(param.shape) == 1:
                classifier_no_decay.append(param)
            else:
                classifier_decay.append(param)
        else:
            if len(param.shape) == 1:
                backbone_no_decay.append(param)
            else:
                backbone_decay.append(param)
    
    optimizer = torch.optim.AdamW([
        {"params": classifier_decay, "lr": classifier_lr, "weight_decay": WEIGHT_DECAY},
        {"params": classifier_no_decay, "lr": classifier_lr, "weight_decay": 0.0},
        {"params": backbone_decay, "lr": backbone_lr, "weight_decay": WEIGHT_DECAY * 2.0},
        {"params": backbone_no_decay, "lr": backbone_lr, "weight_decay": 0.0}
    ], betas=(0.9, 0.999))

    has_validation = valloader is not None

    total_train_loss, total_val_loss = 0.0, 0.0
    total_train_batches, total_val_epochs = 0, 0
    total_train_correct, total_train_samples = 0, 0
    total_val_accuracy = 0.0

    for epoch in range(epochs):
        model.train()
        model.apply(set_bn_eval)
        
        epoch_train_loss, epoch_train_batches = 0.0, 0
        epoch_correct, epoch_samples = 0, 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            
            optimizer.step()

            epoch_train_loss += loss.item()
            epoch_train_batches += 1
            _, predicted = torch.max(outputs.data, 1)
            epoch_samples += labels.size(0)
            epoch_correct += (predicted == labels).sum().item()

        total_train_loss += epoch_train_loss
        total_train_batches += epoch_train_batches
        total_train_correct += epoch_correct
        total_train_samples += epoch_samples

        if has_validation:
            epoch_val_loss, epoch_val_acc = validate_training(model, valloader, criterion, device)
            total_val_loss += epoch_val_loss
            total_val_accuracy += epoch_val_acc
            total_val_epochs += 1

    avg_train_loss = total_train_loss / total_train_batches if total_train_batches > 0 else 0.0
    avg_val_loss = total_val_loss / total_val_epochs if total_val_epochs > 0 else 0.0
    train_accuracy = total_train_correct / total_train_samples if total_train_samples > 0 else 0.0
    val_accuracy = total_val_accuracy / total_val_epochs if total_val_epochs > 0 else 0.0
    
    return avg_train_loss, avg_val_loss, train_accuracy, val_accuracy

# <----------------------------- LOCAL MODEL Testing & Evaluation : Post FL testing of each client model with public test dataset. ----------------------------->
def test(model, testloader, device, return_detailed=False):
    from flwr.common import log
    from logging import INFO, WARNING

    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct, total, total_loss = 0, 0, 0.0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            probs = F.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().tolist())
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    accuracy = correct / total if total > 0 else 0.0
    loss = total_loss / len(testloader) if len(testloader) > 0 else 0.0
    all_preds, all_labels, all_probs = torch.tensor(all_preds), torch.tensor(all_labels), torch.tensor(all_probs)

    # Calculate confusion matrix (0 = Leukemia [Positive class], 1 = Healthy [Negative class])
    TP = ((all_preds == 0) & (all_labels == 0)).sum().item()  # True Positive: Predicted Leukemia, Actual Leukemia
    FP = ((all_preds == 0) & (all_labels == 1)).sum().item()  # False Positive: Predicted Leukemia, Actual Healthy
    FN = ((all_preds == 1) & (all_labels == 0)).sum().item()  # False Negative: Predicted Healthy, Actual Leukemia
    TN = ((all_preds == 1) & (all_labels == 1)).sum().item()  # True Negative: Predicted Healthy, Actual Healthy

    # Standard ML Metrics (Focusing on Leukemia detection)
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0        # Also known as Sensitivity or True Positive Rate
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0   # True Negative Rate
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0     # Positive Predictive Value

    # F1 Score: Harmonic mean of Precision and Recall
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    # Aliases for domain-specific logging and returning
    leukemia_acc = recall
    healthy_acc = specificity
    class_gap = abs(healthy_acc - leukemia_acc)

    log(INFO, f"[EVAL] Overall: {accuracy:.1%} | Leukemia: {leukemia_acc:.1%} | Healthy: {healthy_acc:.1%} | Gap: {class_gap:.1%}")
    if class_gap > 0.3:
        log(WARNING, f"[EVAL] Class imbalance detected! Gap: {class_gap:.1%}")

    if return_detailed:
        return {
            "loss": round(loss, 6), "accuracy": round(accuracy, 6),
            "precision": round(precision, 6), "recall": round(recall, 6),
            "f1_score": round(f1_score, 6), "specificity": round(specificity, 6),
            "leukemia_accuracy": round(leukemia_acc, 6), "healthy_accuracy": round(healthy_acc, 6),
            "class_gap": round(class_gap, 6),
            "confusion_matrix": {"TP": TP, "FP": FP, "FN": FN, "TN": TN},
            "num_samples": total,
            "num_leukemia_samples": (all_labels == 0).sum().item(),
            "num_healthy_samples": (all_labels == 1).sum().item()
        }
    return loss, accuracy

# <----------------------------- KNOWLEDGE DISTILLATION : Distill consensus knowledge into local model using KL divergence with adaptive weighting. ----------------------------->
def distill_knowledge(model, public_loader, consensus_logits, device, epochs, classifier_lr, backbone_lr, temperature,
                      current_round=1, total_rounds=10, adaptive=True):
    base_weight = DISTILL_WEIGHT_BASE
    decay_rate = DISTILL_DECAY_RATE
    weight_decay = WEIGHT_DECAY

    if adaptive and total_rounds > 1:
        progress = (current_round - 1) / (total_rounds - 1)
        adaptive_factor = np.exp(-decay_rate * progress)
        DISTILL_WEIGHT = max(0.20, min(1.0, base_weight * adaptive_factor))
    else:
        DISTILL_WEIGHT = base_weight

    model.to(device)
    model.train()
    model.apply(set_bn_eval)
    
    # Separate parameters into four groups for Discriminative LRs + Weight Decay optimization
    # Rule: Exclude 1D parameters (biases and BN layers) from weight decay.
    classifier_decay = []
    classifier_no_decay = []
    backbone_decay = []
    backbone_no_decay = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
            
        is_classifier = "classifier" in name or "fc" in name
        if is_classifier:
            if len(param.shape) == 1:
                classifier_no_decay.append(param)
            else:
                classifier_decay.append(param)
        else:
            if len(param.shape) == 1:
                backbone_no_decay.append(param)
            else:
                backbone_decay.append(param)
                
    # Optimzer called to updated weights as the model tries to match the conensus value
    optimizer = torch.optim.AdamW([
        {"params": classifier_decay, "lr": classifier_lr, "weight_decay": weight_decay},
        {"params": classifier_no_decay, "lr": classifier_lr, "weight_decay": 0.0},
        {"params": backbone_decay, "lr": backbone_lr, "weight_decay": weight_decay * 2.0},
        {"params": backbone_no_decay, "lr": backbone_lr, "weight_decay": 0.0}
    ], betas=(0.9, 0.999))

    consensus_tensor = torch.from_numpy(consensus_logits).float()

    total_loss, idx = 0.0, 0
    for _epoch in range(epochs):
        for images, _ in public_loader:
            images = images.to(device)
            batch_size = images.size(0)
            if idx + batch_size > len(consensus_tensor):
                break

            batch_consensus = consensus_tensor[idx:idx+batch_size].to(device)

            # Run the public anchor dataset through the client model to obtain the student logits
            student_logits = model(images)

            # Kullback Leibler Divergence
            kl_loss = F.kl_div(
                F.log_softmax(student_logits / temperature, dim=1),
                F.softmax(batch_consensus / temperature, dim=1),
                reduction='batchmean'
            ) * (temperature ** 2)
            loss = DISTILL_WEIGHT * kl_loss

            # Backpropagate and update
            optimizer.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            
            optimizer.step()
            total_loss += loss.item()
            idx += batch_size
        idx = 0

    return total_loss / (len(public_loader) * epochs)

# <----------------------------- CONSENSUS COMPUTATION ----------------------------->
def compute_consensus(
    logits_list: List[np.ndarray], client_metrics: List[Dict], client_configs: List[Dict],
    server_round: int,
) -> Tuple[Optional[np.ndarray], Dict]:
    if not logits_list:
        return None, {"error": "No client logits provided"}

    num_clients = len(logits_list)

    # Weighted aggregation
    raw_weights = []
    for i in range(num_clients):
        # Safely extract num_examples if it exists in the nested dict structure from client_app.py
        if i < len(client_metrics) and "num-examples" in client_metrics[i]:
            raw_weights.append(float(client_metrics[i]["num-examples"]) ** 0.5)
        else:
            raw_weights.append(1.0) # Fallback

    total_samples = sum(raw_weights)
    normalized_weights = [w / total_samples for w in raw_weights]
    
    # Current weighted average
    current_avg_logits = np.average(logits_list, axis=0, weights=normalized_weights)
    
    consensus_logits = current_avg_logits

    weight_breakdown = []
    for i in range(num_clients):
        config = client_configs[i] if i < len(client_configs) else {}
        weight_breakdown.append({
            "client_name": config.get("client_name", f"client_{i}"),
            "model_type": config.get("model_type", "unknown"),
            "normalized_weight": normalized_weights[i],
        })

    aggregation_metadata = {
        "server_round": server_round, "num_clients": num_clients,
        "weight_breakdown": weight_breakdown, "normalized_weights": normalized_weights,
    }

    return consensus_logits, aggregation_metadata

# <----------------------------- FL STRATEGY ----------------------------->
class FLEXMedStrategy(Strategy):
    def __init__(self, config_path: str = CLIENT_INFO_FILE_PATH, batch_size: int = 32):
        super().__init__()
        self.config_path = config_path
        self.batch_size = batch_size
        self.client_configs = load_client_config(config_path)
        self.num_clients = len(self.client_configs)
        self.client_history = {}
        self.total_rounds = 0

# <----------------------------- HELPER METHODS ----------------------------->
    # Evaluate all clients on their private validation sets (pos class is class 0 (all) and healthy is class 1)
    def evaluate_all_clients_on_validation(self, device: torch.device) -> Dict:
        """Evaluate all clients on their private validation sets."""
        from sklearn.metrics import precision_recall_fscore_support

        client_metrics = {}
        for i, client in enumerate(self.client_configs):
            try:
                model = get_model_by_type(client['model_type'])
                model = load_model_weights(model, client['model_path'], device)
                model.to(device)
                _, valloader, _ = load_private_dataset(i, len(self.client_configs), batch_size=self.batch_size)

                if valloader is None:
                    raise ValueError(f"Validation loader for client {i} is None")

                model.eval()
                all_preds, all_labels = [], []
                total_loss = 0.0
                criterion = nn.CrossEntropyLoss()

                with torch.no_grad():
                    for images, labels in valloader:
                        images, labels = images.to(device), labels.to(device)
                        outputs = model(images)
                        total_loss += criterion(outputs, labels).item() * images.size(0)
                        _, preds = torch.max(outputs, 1)
                        all_preds.extend(preds.cpu().numpy())
                        all_labels.extend(labels.cpu().numpy())

                all_preds, all_labels = np.array(all_preds), np.array(all_labels)
                accuracy = (all_preds == all_labels).mean()
                loss = total_loss / len(all_labels)
                precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', pos_label=0, zero_division=0)

                leukemia_mask, healthy_mask = (all_labels == 0), (all_labels == 1)
                leukemia_acc = (all_preds[leukemia_mask] == all_labels[leukemia_mask]).mean() if leukemia_mask.any() else 0
                healthy_acc = (all_preds[healthy_mask] == all_labels[healthy_mask]).mean() if healthy_mask.any() else 0

                client_metrics[str(i)] = {
                    'loss': float(loss), 'accuracy': float(accuracy),
                    'precision': float(precision), 'recall': float(recall), 'f1_score': float(f1),
                    'class_gap': float(abs(leukemia_acc - healthy_acc)),
                    'leukemia_accuracy': float(leukemia_acc), 'healthy_accuracy': float(healthy_acc),
                }
            except Exception as e:
                log(WARNING, f"[VAL] Client {i} validation failed: {e}")
                client_metrics[str(i)] = {"accuracy": None, "loss": None, "error": str(e)}
        return client_metrics

    # Called only once, for the post FL evaluation on unseen test data to evaluate all clients on public test dataset
    def evaluate_all_clients_on_public_test(self, device: torch.device) -> Dict:
        client_metrics = {}
        for i, client in enumerate(self.client_configs):
            try:
                model = get_model_by_type(client['model_type'])
                model = load_model_weights(model, client['model_path'], device)
                model.to(device)
                test_loader = load_public_test_dataset(batch_size=self.batch_size)
                metrics = test(model, test_loader, device, return_detailed=True)
                client_metrics[str(i)] = metrics
            except Exception as e:
                client_metrics[str(i)] = {"accuracy": None, "loss": None, "error": str(e)}
        return client_metrics

# <----------------------------- Start FL simulation ----------------------------->
    def start(self, grid: Grid, initial_arrays: ArrayRecord, num_rounds: int = 3,
              timeout: float = 14400, train_config: Optional[ConfigRecord] = None):

        from flwr.common import log
        from flwr.serverapp.strategy.result import Result
        from logging import INFO, WARNING

        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        train_config = ConfigRecord() if train_config is None else train_config

        result = Result()
        arrays = initial_arrays
        t_start = time.time()
        self.round_metrics_history = {}

        # Best model tracking
        self.best_avg_accuracy = -1.0
        self.best_round = 0

        # <----------------------------- DATA DISTRIBUTION ASSIGNMENT TO EACH CLIENT ----------------------------->
        try:
            log(INFO, "[FL] Pre-initializing Dirichlet partitioner...")
            cache_key = (LOCAL_TRAIN_DATASET_PATH, self.num_clients, DIRICHLET_ALPHA, DIRICHLET_SEED)
            if cache_key not in PARTITIONER_CACHE:
                partitioner, full_dataset = create_dirichlet_partitioner(
                    LOCAL_TRAIN_DATASET_PATH, self.num_clients, DIRICHLET_ALPHA, DIRICHLET_SEED
                )
                PARTITIONER_CACHE[cache_key] = (partitioner, full_dataset)
                log(INFO, f"[FL] Partitioner initialized: {self.num_clients} partitions, alpha={DIRICHLET_ALPHA}")
            
            save_all_clients_data_heterogeneity(self.client_configs, PARTITIONER_CACHE) # Save data heterogeneity for all clients at FL start
        except Exception as e:
            log(WARNING, f"[FL] Failed to pre-initialize partitioner: {e}")

        print_client_data_distribution_summary(self.client_configs, PARTITIONER_CACHE)

        # <----------------------------- TRAINING STARTS BASED ON THE NUMBER OF ROUNDS ----------------------------->
        for current_round in range(1, num_rounds + 1):
            log(INFO, f"\n{'='*70}\n[ROUND {current_round}/{num_rounds}]\n{'='*70}")

            # <----------------------------- Configure and aggregate training ----------------------------->
            train_msgs = self.configure_train(current_round, arrays, train_config, grid)
            train_replies = grid.send_and_receive(messages=train_msgs, timeout=timeout)
            agg_arrays, agg_metrics, training_metrics = self.aggregate_train(current_round, train_replies) # agg_arrays is the new consensus which will then be passed back to the clients

            if agg_arrays is not None:
                result.arrays = agg_arrays
                arrays = agg_arrays
            if agg_metrics:
                result.train_metrics_clientapp[current_round] = agg_metrics
            if training_metrics:
                try:
                    save_round_training_metrics(current_round, training_metrics, self.client_configs, PARTITIONER_CACHE)
                except Exception as e:
                    log(WARNING, f"[ROUND {current_round}] Failed to save training metrics: {e}")

            try:
                # <----------------------------- Evaluate model on validation set ----------------------------->
                round_val_metrics = self.evaluate_all_clients_on_validation(device)
                save_round_validation_metrics(current_round, round_val_metrics, self.client_configs)
                self.round_metrics_history[f"round_{current_round}_validation"] = round_val_metrics

                for client_id, metrics in round_val_metrics.items():
                    if client_id not in self.client_history:
                        self.client_history[client_id] = []
                    self.client_history[client_id].append({'round': current_round, 'metrics': metrics})

                check_for_degradation_warnings(round_val_metrics, current_round, self.client_history)

                # <----------------------------- Best model checkpoint tracking ----------------------------->
                valid_accs = [
                    (m.get('leukemia_accuracy', 0) + m.get('healthy_accuracy', 0)) / 2
                    for m in round_val_metrics.values()
                    if m.get('leukemia_accuracy') is not None and m.get('healthy_accuracy') is not None
                ]
                if valid_accs:
                    avg_acc = sum(valid_accs) / len(valid_accs)
                    if avg_acc > self.best_avg_accuracy:
                        self.best_avg_accuracy = avg_acc
                        self.best_round = current_round
                        log(INFO, f"[BEST] New best avg balanced accuracy {avg_acc:.2%} at round {current_round}")
                        for client_cfg in self.client_configs:
                            src = client_cfg.get('model_path', '')
                            if src and os.path.exists(src):
                                base, ext = os.path.splitext(src)
                                dst = f"{base}_best{ext}"
                                shutil.copy2(src, dst)
                    else:
                        log(INFO, f"[BEST] Round {current_round} avg balanced accuracy {avg_acc:.2%} "
                                  f"(best: {self.best_avg_accuracy:.2%} at round {self.best_round})")

            except Exception as e:
                log(WARNING, f"[ROUND {current_round}] Validation evaluation failed: {e}")

        # <----------------------------- Restore best model checkpoints before final evaluation ----------------------------->
        if self.best_round > 0 and self.best_round < num_rounds:
            log(INFO, f"[BEST] Restoring best models from round {self.best_round} "
                      f"(avg accuracy: {self.best_avg_accuracy:.2%})")
            for client_cfg in self.client_configs:
                src = client_cfg.get('model_path', '')
                if src:
                    base, ext = os.path.splitext(src)
                    best_path = f"{base}_best{ext}"
                    if os.path.exists(best_path):
                        shutil.copy2(best_path, src)
                        log(INFO, f"[BEST] Restored {os.path.basename(src)} from round {self.best_round}")
        else:
            log(INFO, f"[BEST] Final round {num_rounds} was the best — no restore needed")

        log(INFO, f"\n{'='*70}\n[GLOBAL] Final Federated Model Evaluation\n{'='*70}")

        try:
            # <----------------------------- Evaluate model on public test set (unseen data) ----------------------------->
            global_post_fl_metrics = self.evaluate_all_clients_on_public_test(device)
            save_global_post_fl_metrics(global_post_fl_metrics, self.client_configs, self.best_round)
        except Exception as e:
            log(WARNING, f"[GLOBAL] Post-FL Evaluation failed: {e}")

        log(INFO, f"Strategy execution finished in {time.time() - t_start:.2f}s")
        return result

    # <----------------------------- Aggregate training results ----------------------------->
    def aggregate_train(self, server_round: int, results: Iterable[Message], **kwargs):
        results_list = list(results)
        if not results_list:
            return None, {}, {}

        # FIX: Sort the results by client_id to prevent metrics from swapping between clients
        results_list.sort(key=lambda msg: msg.content.get("metrics", {}).get("client_id", 999) if msg.has_content() else 999)

        logits_list, client_metrics_list, client_names = [], [], []

        for i, msg in enumerate(results_list):
            if not msg.has_content():
                continue
            try:
                client_logits = msg.content["arrays"]["0"].numpy()
                logits_list.append(client_logits)
                client_metrics_list.append(msg.content.get("metrics", {}))
                if i < len(self.client_configs):
                    client_names.append(self.client_configs[i]['client_name'])
            except (KeyError, IndexError):
                pass

        consensus_logits, aggregation_metadata = compute_consensus(
            logits_list, client_metrics_list, self.client_configs[:len(logits_list)],
            server_round, 
        )

        if consensus_logits is None:
            return None, {}, {}

        metrics_aggregated = {
            "consensus_round": server_round, "num_clients": len(logits_list),
            "client_names": client_names, "weights": aggregation_metadata['normalized_weights'],
            "aggregation_details": aggregation_metadata
        }

        training_metrics_for_persistence = extract_training_metrics_for_persistence(
            client_metrics_list, aggregation_metadata, self.client_configs
        )

        return ArrayRecord([consensus_logits]), metrics_aggregated, training_metrics_for_persistence

    def configure_train(self, server_round: int, arrays: Optional[ArrayRecord],
        config: ConfigRecord, grid: Grid) -> Iterable[Message]:
        messages = []
        for node_id in grid.get_node_ids():
            msg = Message(
                metadata=Metadata(run_id=0, message_id=str(uuid.uuid4()), src_node_id=0, dst_node_id=node_id,
                                  reply_to_message_id="", group_id=str(server_round), ttl=86400.0,
                                  message_type="train", created_at=time.time()),
                content=RecordDict({"arrays": arrays, "config": config}),
            )
            messages.append(msg)
        return messages

    def configure_evaluate(self, server_round: int, arrays, config, grid) -> Iterable[Message]:
        return []

    def aggregate_evaluate(self, server_round: int, results: Iterable[Message], **kwargs):
        return None, None

    def summary(self) -> str:
        return ""

---
## 5. Server App (`flex_med/server_app.py`)

In [ ]:
%%writefile flex_med/server_app.py
import numpy as np
import json
import os
from flwr.app import ArrayRecord, ConfigRecord, Context
from flwr.serverapp import ServerApp
from flwr.server import Grid
from flex_med.utils.config import CLIENT_INFO_FILE_PATH, MODEL_CHECKPOINT_FILE_PATH
from flex_med.task import (
    FLEXMedStrategy, load_public_dataset, NUM_CLASSES,
    load_client_config
)

app = ServerApp()

@app.main()
def main(grid: Grid, context: Context) -> None:

    # <-------------------------------------- LOAD CONFIGURATIONS -------------------------------------->
    num_rounds: int = context.run_config["num-server-rounds"]
    lr: float = context.run_config["lr"]
    local_epochs: int = context.run_config.get("local-epochs", 1)
    batch_size: int = context.run_config.get("batch-size", 32)

    print(f"\n[SERVER] Starting FL ({num_rounds} rounds, lr={lr})")

    # <-------------------------------------- LOAD CLIENT CONFIGURATIONS -------------------------------------->
    client_configs = load_client_config()
    print(f"[SERVER] {len(client_configs)} clients loaded")

    # <-------------------------------------- LOAD PUBLIC ANCHOR DATASET -------------------------------------->
    public_loader = load_public_dataset(batch_size=batch_size)
    num_samples = len(public_loader.dataset)

    initial_consensus = np.zeros((num_samples, NUM_CLASSES), dtype=np.float32) # creates an array of arrays [[0,0] * num_samples]

    # <-------------------------------------- LOAD FL STRATEGY -------------------------------------->
    strategy = FLEXMedStrategy(
        config_path=CLIENT_INFO_FILE_PATH,
        batch_size=batch_size
    )

    # <-------------------------------------- EXECUTE FL SIMULATION -------------------------------------->
    print(f"\n{'='*60}")
    print(f"[SERVER] Executing Federated Learning")
    print(f"{'='*60}\n")

    result = strategy.start(
        grid=grid,
        initial_arrays=ArrayRecord([initial_consensus]),
        train_config=ConfigRecord({"lr": lr, "local_epochs": local_epochs}),
        num_rounds=num_rounds,
    )

    # <-------------------------------------- SAVE RESULTS -------------------------------------->
    print(f"\n{'='*60}")
    print(f"[SERVER] Federated Learning Complete")
    print(f"{'='*60}\n")

    checkpoint_dir = os.path.dirname(MODEL_CHECKPOINT_FILE_PATH)
    consensus_save_path = os.path.join(checkpoint_dir, "final_consensus.npy")
    metrics_save_path = os.path.join(checkpoint_dir, "training_summary.json")

    final_logits = None
    try:
        final_logits = result.arrays["0"].numpy()
        np.save(consensus_save_path, final_logits)
        print(f"  Consensus saved: {consensus_save_path}")
    except Exception as e:
        print(f"  Error saving consensus: {e}")

    summary = {
        "num_rounds": num_rounds,
        "num_clients": strategy.num_clients,
        "learning_rate": lr,
        "local_epochs": local_epochs,
        "public_dataset_size": num_samples,
        "num_classes": NUM_CLASSES,
        "clients": [
            {"id": i, "name": c['client_name'], "model_type": c['model_type'], "model_path": c['model_path']}
            for i, c in enumerate(client_configs)
        ]
    }

    try:
        with open(metrics_save_path, 'w') as f:
            json.dump(summary, f, indent=2)
        print(f"  Summary saved: {metrics_save_path}")
    except Exception as e:
        print(f"  Error saving summary: {e}")

    print(f"\n[SERVER] Training complete!\n")

---
## 6. Client App (`flex_med/client_app.py`)

In [ ]:
%%writefile flex_med/client_app.py
import torch
import os
import time
import numpy as np
from flwr.app import ArrayRecord, Context, Message, MetricRecord, RecordDict
from flwr.clientapp import ClientApp
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="torchvision.transforms._functional_pil")
warnings.filterwarnings("ignore", message=".*'mode' parameter is deprecated.*")
import random
from flex_med.utils.config import CLIENT_INFO_FILE_PATH, DIRICHLET_SEED
from flex_med.task import (
    load_private_dataset,
    load_public_dataset, get_public_logits, distill_knowledge,
    train as train_fn, NUM_CLASSES
)
from flex_med.utils.helpers import (
    apply_freeze_strategy, load_model_weights,
    get_display_id, save_model, load_model_for_client,
    compute_per_class_accuracy, freeze_backbone, unfreeze_fraction_of_last_block
)

app = ClientApp()

@app.train()
def train(msg: Message, context: Context):
    # <-------------------------------------- LOAD CONFIGURATIONS -------------------------------------->
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]
    total_rounds = context.run_config["num-server-rounds"]

    try:
        server_round = int(msg.metadata.group_id)
    except (ValueError, TypeError, AttributeError):
        server_round = 1

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # Deterministic seeding for reproducibility
    random.seed(DIRICHLET_SEED)
    np.random.seed(DIRICHLET_SEED)
    torch.manual_seed(DIRICHLET_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(DIRICHLET_SEED)

    print(f"\n{'='*60}")
    
    model, model_path, client_config = load_model_for_client(partition_id) # load model with the partition assigned to it
    display_id = get_display_id(partition_id, client_config)

    print(f"[{display_id} (Partition {partition_id})] ROUND {server_round} - Training Phase")
    print(f"{'='*60}")
    model_type = client_config['model_type']

    os.makedirs(os.path.dirname(model_path), exist_ok=True)

    if os.path.exists(model_path):
        model = load_model_weights(model, model_path, device)
        print(f"[{display_id}] Loaded model weights")

    model.to(device)

    # <-------------------------------------- FREEZE/UNFREEZE STRATEGY -------------------------------------->
    base_lr = msg.content["config"]["lr"]
    progress = server_round / total_rounds

    # Applying the 2-stage unfreeze strategy
    model = apply_freeze_strategy(model, model_type, server_round, total_rounds)

    # Discriminative learning rates — Phase transitions
    from flex_med.utils.helpers import PHASE2_START_ROUND

    phase = "INITIALIZING"

    if server_round < PHASE2_START_ROUND:
        # Phase 1: Classifier Head Only (R1-4)
        classifier_lr = base_lr
        backbone_lr = 0.0
        phase = "PHASE 1 - Classifier Head Only (R1-4)"
    else:
        # Phase 2: 50% Backbone Refinement (R5+)
        classifier_lr = base_lr * 0.6
        backbone_lr = base_lr * 0.03
        phase = "PHASE 2 - 50% Backbone Refinement (R5+)"

    print(f"[{display_id}] [Strategy] Round {server_round}/{total_rounds}: {phase}")
    print(f"[{display_id}] Round {server_round}/{total_rounds} ({progress:.0%})")
    print(f"[{display_id}] Classifier LR: {classifier_lr:.6f} | Backbone LR: {backbone_lr:.6f} (dynamic)")

    # <-------------------------------------- KNOWLEDGE DISTILLATION -------------------------------------->
    distill_loss = 0.0
    if "arrays" in msg.content and msg.content["arrays"]:
        try:
            consensus_logits = msg.content["arrays"]["0"].numpy()
            if consensus_logits.shape[0] > 0:
                print(f"[{display_id}] Phase 1: Knowledge Distillation")

                #  Load public dataset for logit generation
                public_loader = load_public_dataset(batch_size=context.run_config["batch-size"])

                #  Perform knowledge distillation
                distill_loss = distill_knowledge(
                    model=model, public_loader=public_loader, consensus_logits=consensus_logits,
                    device=device, epochs=context.run_config["distill-epochs"], 
                    classifier_lr=classifier_lr, backbone_lr=backbone_lr, temperature=context.run_config["temperature"],
                    current_round=server_round, total_rounds=total_rounds, adaptive=True
                )
                print(f"[{display_id}] Distillation Loss: {distill_loss:.4f}")
        except Exception as e:
            print(f"[Client {partition_id}] Distillation failed: {e}")

    # <-------------------------------------- PRIVATE TRAINING -------------------------------------->
    print(f"[{display_id}] Private Training")

    trainloader, valloader, _ = load_private_dataset(partition_id, num_partitions, batch_size=context.run_config["batch-size"])
    if trainloader is None:
        raise ValueError(f"[{display_id}] No training data available")

    dataset_len = len(trainloader.dataset)

    start_time = time.time()
    train_loss, val_loss, train_accuracy, val_accuracy = train_fn(
        model=model, trainloader=trainloader, epochs=context.run_config["local-epochs"],
        classifier_lr=classifier_lr, backbone_lr=backbone_lr, device=device,
        valloader=valloader, server_round=server_round, total_rounds=total_rounds
    )
    training_time = time.time() - start_time

    print(f"[{display_id}] Train Loss: {train_loss:.4f}, Acc: {train_accuracy:.2%}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2%} ({dataset_len} samples, {training_time:.1f}s)")

    # <-------------------------------------- COMPUTE BALANCED ACCURACY -------------------------------------->
    balanced_accuracy = compute_per_class_accuracy(model, valloader, device)
    print(f"[{display_id}] Balanced Accuracy: {balanced_accuracy:.2%}")

    # <-------------------------------------- SAVE MODEL -------------------------------------->
    try:
        save_model(model, model_path, model_type)
        print(f"[{display_id}] Model saved")
    except Exception as e:
        print(f"[{display_id}] Save failed: {e}")

    # <-------------------------------------- GENERATE PUBLIC LOGITS -------------------------------------->
    public_loader = load_public_dataset(batch_size=context.run_config["batch-size"])
    public_logits = get_public_logits(model, public_loader, device)

    print(f"[{display_id}] Round {server_round} Complete\n")

    # <-------------------------------------- RETURN METRICS -------------------------------------->
    return Message(
        content=RecordDict({
            "arrays": ArrayRecord([public_logits]),
            "metrics": MetricRecord({
                "train_loss": train_loss,
                "train_accuracy": train_accuracy,
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
                "balanced_accuracy": balanced_accuracy,
                "distill_loss": distill_loss,
                "num-examples": dataset_len,
                "training_time": training_time,
                "client_id": partition_id,
            })
        }),
        reply_to=msg
    )

---
## 7. Flower Configuration (`pyproject.toml`)

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "flex_med"
version = "2.0.0"
description = "Federated Learning for Medical Image Classification"
license = "Apache-2.0"
dependencies = [
    "flwr[simulation]>=1.23.0",
    "flwr-datasets[vision]>=0.5.0",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "matplotlib>=3.7.0",
    "pandas>=2.0.0",
    "numpy>=1.24.0",
    "pillow>=10.0.0",
    "scikit-learn>=1.3.0",
    "requests>=2.28.0",
    "grad-cam>=1.4.8",
    "lime>=0.2.0.1",
    "scikit-image>=0.21.0",
]

[tool.flwr.app]
publisher = "shrenik"

[tool.flwr.app.components]
serverapp = "flex_med.server_app:app"
clientapp = "flex_med.client_app:app"

[tool.flwr.app.config]
num-server-rounds = 10
fraction-train = 1.0
fraction-evaluate = 1.0
local-epochs = 2
batch-size = 32
lr = 0.0005
lr-decay = 0.9
distill-lr = 0.0005
distill-epochs = 2
temperature = 3.0

[tool.flwr.federations]
default = "local-simulation"

[tool.flwr.federations.local-simulation.options]
num-supernodes = 3
num-partitions = 3

[tool.flwr.federations.remote-federation]
address = "<SUPERLINK-ADDRESS>:<PORT>"
insecure = true

[tool.flwr.app.federations.default.options]
num-supernodes = 3
num-partitions = 3

[tool.hatch.build.targets.wheel]
packages = ["."]

---
## 8. Health Verification

Validates dataset structure, client configs, and environment before running FL.

In [ ]:
import os
import json
from pathlib import Path

# Quick health check
from flex_med.utils.config import (
    PUBLIC_ANCHOR_DATASET_PATH, PUBLIC_TEST_DATASET_PATH,
    LOCAL_TRAIN_DATASET_PATH, CLIENT_INFO_FILE_PATH,
    DIRICHLET_ALPHA, DIRICHLET_SEED, DIRICHLET_MIN_PARTITION_SIZE
)

print("=" * 60)
print("FLEX-Med FL Health Check")
print("=" * 60)

# Check datasets
for name, path in [("Public Anchor", PUBLIC_ANCHOR_DATASET_PATH),
                    ("Public Test", PUBLIC_TEST_DATASET_PATH),
                    ("Local Train", LOCAL_TRAIN_DATASET_PATH)]:
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    count = ""
    if exists:
        total = sum(len(files) for _, _, files in os.walk(path))
        count = f" ({total} files)"
    print(f"  [{status}] {name}: {path}{count}")

# Check client config
print(f"\n  Config file: {CLIENT_INFO_FILE_PATH}")
if os.path.exists(CLIENT_INFO_FILE_PATH):
    with open(CLIENT_INFO_FILE_PATH) as f:
        data = json.load(f)
    clients = data.get("clients", data) if isinstance(data, dict) else data
    print(f"  [OK] {len(clients)} clients configured")
    for c in clients:
        print(f"    - {c.get('client_name', 'Unknown')}: {c.get('model_type', '?')}")
else:
    print("  [MISSING] Client config file not found")

print(f"\n  Dirichlet: alpha={DIRICHLET_ALPHA}, seed={DIRICHLET_SEED}, min_partition={DIRICHLET_MIN_PARTITION_SIZE}")
print("=" * 60)

---
## 9. API Definition

FastAPI server that accepts `/start_fl` requests and runs the FL simulation.

In [ ]:
import os
import sys
import json
import logging
import subprocess
from typing import List, Tuple, Optional
import re

from fastapi import FastAPI, BackgroundTasks, HTTPException
from pydantic import BaseModel
from starlette.middleware.cors import CORSMiddleware

from flex_med.utils.config import (
    PROJECT_DIR,
    SIMULATION_LOG_PATH,
    CLIENT_INFO_FILE_PATH,
    PYPROJECT_PATH,
    RUN_SIMULATION_SHELL_FILE_PATH,
)

os.makedirs(PROJECT_DIR, exist_ok=True)

# Logging
logger = logging.getLogger("inference_logger")
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()
stream_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.info("=== Flex-Med Orchestrator Started ===")

# Validation
def validate_client_config(client) -> Tuple[bool, str]:
    valid_models = ['efficientnet_b0', 'efficientnet_b1', 'efficientnet_b2']
    if client.model_type not in valid_models:
        return False, f"Invalid model type: {client.model_type}. Must be one of {valid_models}"
    if not client.model_path:
        return False, f"Client {client.client_name}: model_path is required"
    model_dir = os.path.dirname(client.model_path)
    if not os.path.exists(model_dir):
        try:
            os.makedirs(model_dir, exist_ok=True)
            logger.info(f"Created model directory: {model_dir}")
        except Exception as e:
            return False, f"Cannot create model directory: {model_dir}, error: {e}"
    return True, ""

# Pydantic models
class ClientConfig(BaseModel):
    id: int
    client_name: str
    model_type: str
    model_path: str
    status: Optional[str] = "Active"
    created_at: Optional[str] = None
    metrics: Optional[str] = None

class FLRequest(BaseModel):
    """Request format matching what the backend sends (fl_simulation.py)."""
    simulation_id: int
    clients: List[ClientConfig]
    supabase_url: str
    supabase_key: str
    fl_config: Optional[dict] = None
    training_config: Optional[dict] = None
    dataset_paths: Optional[dict] = None

class SimulationStatus(BaseModel):
    status: str
    message: str
    details: Optional[dict] = None

# State
simulation_state = {"status": "idle", "process": None, "log_file": None}

# FastAPI app
app = FastAPI(title="FLEX-Med FL Orchestrator")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def update_pyproject_toml(num_clients: int, fl_config: Optional[dict] = None, path: str = PYPROJECT_PATH) -> bool:
    """Update pyproject.toml with correct num-supernodes and FL parameters."""
    try:
        if not os.path.exists(path):
            logger.error(f"pyproject.toml not found at {path}")
            return False

        with open(path, 'r') as f:
            content = f.read()

        # Update num-supernodes
        pattern = r'(num-supernodes\s*=\s*)\d+'
        if not re.search(pattern, content):
            logger.warning("num-supernodes not found in pyproject.toml")
            return False

        old_match = re.search(r'num-supernodes\s*=\s*(\d+)', content)
        old_value = old_match.group(1) if old_match else "unknown"
        matches = re.findall(pattern, content)

        new_content = re.sub(pattern, f'\\g<1>{num_clients}', content)

        # Update num-partitions to match
        partitions_pattern = r'(num-partitions\s*=\s*)\d+'
        new_content = re.sub(partitions_pattern, f'\\g<1>{num_clients}', new_content)

        # Update FL config parameters in [tool.flwr.app.config] section
        if fl_config:
            param_mapping = {
                "num_server_rounds": "num-server-rounds",
                "local_epochs": "local-epochs",
                "batch_size": "batch-size",
                "lr": "lr",
                "lr_decay": "lr-decay",
                "distill_lr": "distill-lr",
                "distill_epochs": "distill-epochs",
                "temperature": "temperature",
            }
            for config_key, toml_key in param_mapping.items():
                if config_key in fl_config:
                    value = fl_config[config_key]
                    fl_pattern = rf'({re.escape(toml_key)}\s*=\s*)[\d.]+'
                    if re.search(fl_pattern, new_content):
                        new_content = re.sub(fl_pattern, f'\\g<1>{value}', new_content)
                        logger.info(f"Updated pyproject.toml: {toml_key} = {value}")
                    else:
                        logger.warning(f"Key '{toml_key}' not found in pyproject.toml")

        with open(path, 'w') as f:
            f.write(new_content)

        # Verify
        with open(path, 'r') as f:
            verify_content = f.read()
        verify_match = re.search(r'num-supernodes\s*=\s*(\d+)', verify_content)
        if verify_match and int(verify_match.group(1)) != num_clients:
            logger.error(f"Write verification failed! Expected {num_clients}, got {verify_match.group(1)}")
            return False

        logger.info(f"Updated pyproject.toml: num-supernodes {old_value} -> {num_clients} ({len(matches)} occurrences)")
        if fl_config:
            logger.info(f"Updated pyproject.toml FL params: {list(fl_config.keys())}")
        return True

    except Exception as e:
        logger.error(f"Failed to update pyproject.toml: {e}")
        return False

def create_run_script(simulation_id: int, num_clients: int, supabase_url: str, supabase_key: str,
                      training_config: Optional[dict] = None):
    """Create shell script that runs the Flower simulation with env vars and pre-flight checks."""
    # Build training config env var exports
    training_env_exports = ""
    if training_config:
        env_mapping = {
            "dirichlet_alpha": "FLEX_MED_DIRICHLET_ALPHA",
            "dirichlet_seed": "FLEX_MED_DIRICHLET_SEED",
            "dirichlet_min_partition_size": "FLEX_MED_DIRICHLET_MIN_PARTITION_SIZE",
            "minority_boost": "FLEX_MED_MINORITY_BOOST",
            "distill_weight_base": "FLEX_MED_DISTILL_WEIGHT",
            "distill_decay_rate": "FLEX_MED_DISTILL_DECAY",
            "train_loss_weight": "FLEX_MED_TRAIN_LOSS_WEIGHT",
            "distill_loss_weight": "FLEX_MED_DISTILL_LOSS_WEIGHT",
        }
        for config_key, env_var in env_mapping.items():
            if config_key in training_config:
                training_env_exports += f'export {env_var}="{training_config[config_key]}"\n'

    script_content = f"""#!/bin/bash
cd "{PROJECT_DIR}"

echo "[Script] Starting Flex-Med Simulation at $(date)..."
echo "[Script] Simulation ID: {simulation_id}"

# Export environment variables for FL system
export FLEX_MED_SIMULATION_ID="{simulation_id}"
export SUPABASE_URL="{supabase_url}"
export SUPABASE_KEY="{supabase_key}"

# Training strategy configuration (from system_config preset)
{training_env_exports}
# Dataset paths configured via config.py defaults

# Optional: Configure Ray memory settings
export RAY_memory_usage_threshold="0.98"

echo "[Script] Training config env vars exported"
echo "[Script] FLEX_MED_DIRICHLET_ALPHA=$FLEX_MED_DIRICHLET_ALPHA"
echo "[Script] FLEX_MED_MINORITY_BOOST=$FLEX_MED_MINORITY_BOOST"
echo "[Script] FLEX_MED_DISTILL_WEIGHT=$FLEX_MED_DISTILL_WEIGHT"

# Only install the local package in editable mode (Fast & Lightweight)
pip install -e . -q

# ==========================================
# PRE-FLIGHT: Verify num-supernodes matches client count
# ==========================================
sync  # Force filesystem flush

EXPECTED_CLIENTS={num_clients}
ACTUAL_SUPERNODES=$(grep -oP 'num-supernodes\\s*=\\s*\\K\\d+' pyproject.toml | head -1)
DATA_JSON_CLIENTS=$(python3 -c "
import json
d = json.load(open('{CLIENT_INFO_FILE_PATH}'))
clients = d.get('clients', d) if isinstance(d, dict) else d
print(len(clients))
")

echo "[Pre-flight] Expected clients:       $EXPECTED_CLIENTS"
echo "[Pre-flight] pyproject num-supernodes: $ACTUAL_SUPERNODES"
echo "[Pre-flight] client_data.json clients: $DATA_JSON_CLIENTS"

if [ "$ACTUAL_SUPERNODES" != "$EXPECTED_CLIENTS" ]; then
    echo "[Pre-flight] MISMATCH detected! Fixing pyproject.toml..."
    sed -i "s/num-supernodes = [0-9]*/num-supernodes = $EXPECTED_CLIENTS/g" pyproject.toml
    sync
    NEW_VALUE=$(grep -oP 'num-supernodes\\s*=\\s*\\K\\d+' pyproject.toml | head -1)
    echo "[Pre-flight] Fixed: num-supernodes = $NEW_VALUE"
fi

if [ "$DATA_JSON_CLIENTS" != "$EXPECTED_CLIENTS" ]; then
    echo "[Pre-flight] ERROR: client_data.json has $DATA_JSON_CLIENTS clients but expected $EXPECTED_CLIENTS"
    echo "[Pre-flight] Aborting simulation."
    exit 1
fi

echo "[Pre-flight] All checks passed. Starting FL..."

# Run Flower CLI
flwr run .
"""

    with open(RUN_SIMULATION_SHELL_FILE_PATH, "w") as f:
        f.write(script_content)
    os.chmod(RUN_SIMULATION_SHELL_FILE_PATH, 0o755)
    logger.info(f"Run script created at {RUN_SIMULATION_SHELL_FILE_PATH}")
    return RUN_SIMULATION_SHELL_FILE_PATH

def update_data_json(simulation_id: int, clients: List[ClientConfig], path: str = CLIENT_INFO_FILE_PATH):
    """Save client configuration to data.json with simulation metadata."""
    try:
        client_dicts = [c.model_dump() for c in clients]
        config_data = {
            "simulation_id": simulation_id,
            "clients": client_dicts
        }
        with open(path, 'w') as f:
            json.dump(config_data, f, indent=2)
        logger.info(f"Updated data.json with simulation_id={simulation_id} and {len(clients)} clients")
        return True
    except Exception as e:
        logger.error(f"Failed to update data.json: {e}")
        return False

def run_simulation_background(script_path: str, log_path: str):
    """Run the FL simulation shell script in the background."""
    try:
        simulation_state["status"] = "running"
        logger.info(f"Starting FL simulation via script: {script_path}")

        os.makedirs(os.path.dirname(log_path), exist_ok=True)

        with open(log_path, 'w') as log_file:
            process = subprocess.Popen(
                ["bash", script_path],
                cwd=PROJECT_DIR,
                stdout=log_file,
                stderr=subprocess.STDOUT,
            )
            simulation_state["process"] = process
            simulation_state["log_file"] = log_path
            process.wait()

            if process.returncode == 0:
                simulation_state["status"] = "completed"
                logger.info("FL simulation completed successfully")
            else:
                simulation_state["status"] = "failed"
                logger.error(f"FL simulation failed with code {process.returncode}")

    except Exception as e:
        simulation_state["status"] = "failed"
        logger.error(f"Simulation error: {e}")

@app.get("/health")
async def health():
    return {"status": "healthy", "service": "FLEX-Med FL Orchestrator"}

@app.post("/start_fl")
async def start_fl(request: FLRequest, background_tasks: BackgroundTasks):
    if simulation_state["status"] == "running":
        raise HTTPException(status_code=409, detail="Simulation already running")

    simulation_id = request.simulation_id
    clients = request.clients
    num_clients = len(clients)

    # Validate clients
    for client in clients:
        is_valid, error = validate_client_config(client)
        if not is_valid:
            raise HTTPException(status_code=400, detail=error)

    logger.info(f"All {num_clients} client configurations validated")

    # Step 1: Update data.json
    logger.info("\n[Endpoint] Step 1: Updating data.json...")
    if not update_data_json(simulation_id, clients):
        raise HTTPException(status_code=500, detail="Failed to save client configuration to data.json")

    # Step 2: Update pyproject.toml (num-supernodes + FL params)
    logger.info("\n[Endpoint] Step 2: Updating pyproject.toml...")
    if not update_pyproject_toml(num_clients, request.fl_config):
        raise HTTPException(
            status_code=500,
            detail=f"Failed to update pyproject.toml num-supernodes to {num_clients}"
        )

    # Step 3: Create run script (with Supabase creds + training config env vars + pre-flight)
    logger.info("\n[Endpoint] Step 3: Creating run script...")
    logger.info(f"  - FLEX_MED_SIMULATION_ID={simulation_id}")
    logger.info(f"  - SUPABASE_URL={request.supabase_url}")
    logger.info(f"  - SUPABASE_KEY=***hidden***")
    if request.training_config:
        logger.info(f"  - Training config env vars: {list(request.training_config.keys())}")

    script_path = create_run_script(
        simulation_id=simulation_id,
        num_clients=num_clients,
        supabase_url=request.supabase_url,
        supabase_key=request.supabase_key,
        training_config=request.training_config,
    )

    # Step 4: Set environment variables (for current process too)
    os.environ['FLEX_MED_SIMULATION_ID'] = str(simulation_id)
    os.environ['SUPABASE_URL'] = request.supabase_url
    os.environ['SUPABASE_KEY'] = request.supabase_key

    client_config_json = json.dumps({
        "simulation_id": simulation_id,
        "clients": [c.model_dump() for c in clients]
    })
    os.environ['FLEX_MED_CLIENT_CONFIGS'] = client_config_json

    # Step 5: Trigger simulation
    logger.info("\n[Endpoint] Step 5: Starting FL simulation...")
    logger.info(f"  - Number of clients: {num_clients}")
    logger.info(f"  - All clients use Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH")

    background_tasks.add_task(run_simulation_background, script_path, SIMULATION_LOG_PATH)

    return {
        "status": "started",
        "message": f"FL simulation {simulation_id} started with {num_clients} clients",
        "simulation_id": simulation_id,
        "log_path": SIMULATION_LOG_PATH,
        "version": "3.1"
    }

@app.get("/simulation/status")
async def get_simulation_status():
    response = {"status": simulation_state["status"]}
    if simulation_state["status"] == "running" and simulation_state.get("process"):
        poll = simulation_state["process"].poll()
        if poll is not None:
            simulation_state["status"] = "completed" if poll == 0 else "failed"
            response["status"] = simulation_state["status"]
    if simulation_state.get("log_file") and os.path.exists(simulation_state["log_file"]):
        try:
            with open(simulation_state["log_file"], 'r') as f:
                lines = f.readlines()
                response["last_log_lines"] = [l.strip() for l in lines[-20:]]
        except Exception:
            pass
    return response

@app.post("/simulation/stop")
async def stop_simulation():
    if simulation_state["status"] != "running" or not simulation_state.get("process"):
        return {"status": "not_running", "message": "No simulation is currently running"}
    try:
        simulation_state["process"].terminate()
        simulation_state["status"] = "stopped"
        return {"status": "stopped", "message": "Simulation terminated"}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to stop: {e}")

logger.info("Flex-Med Orchestrator Ready (v3.1)")
logger.info("  Endpoint: POST /start_fl")
logger.info("  Schema: {simulation_id, clients, supabase_url, supabase_key, fl_config?, training_config?}")
logger.info("  All clients use Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH")
logger.info("  FL params written to pyproject.toml, training params exported as env vars")


---
## 10. ngrok Setup & Server Start

**IMPORTANT**: Replace `AUTH_TOKEN` with your ngrok auth token.

In [ ]:
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok
from pyngrok.process import kill_process

# Cleanup previous tunnels
try:
    ngrok.disconnect()
    kill_process()
except:
    pass

AUTH_TOKEN = "38BI51ntfzceNwdU1BTYDLnAZib_3S3SmcwQJFykk89BGBmEJE"
ngrok.set_auth_token(AUTH_TOKEN)

http_tunnel = ngrok.connect(8000)
public_url = http_tunnel.public_url

print(f"\n{'='*60}")
print(f"  API URL: {public_url}")
print(f"  Start FL: POST {public_url}/start_fl")
print(f"  Status:   GET  {public_url}/simulation/status")
print(f"  Health:   GET  {public_url}/health")
print(f"{'='*60}\n")

nest_asyncio.apply()

def run_uvicorn():
    uvicorn.run(app, port=8000, host="0.0.0.0")

thread = threading.Thread(target=run_uvicorn)
thread.start()

---
## 11. View Training Logs

In [ ]:
from flex_med.utils.config import SIMULATION_LOG_PATH
!tail -f "{SIMULATION_LOG_PATH}"

---
## 12. Kill Server Process

In [ ]:
!fuser -k 8000/tcp
print("Killed processes on port 8000. Re-run the ngrok cell to restart.")